# Part 1 · Data Preparation

**Recipe Rating Analysis with PySpark** — notebook 1 of 3

This notebook loads the raw recipe and interaction data from Amazon S3, cleans and restructures it with PySpark, and saves an interaction-level dataset for exploratory analysis.

| Step | What happens |
|---|---|
| 1 | Load ~232K recipes |
| 2 | Split the `nutrition` string into seven numeric columns |
| 3 | Standardise nutrition values per 100 calories |
| 4 | Convert `tags` from a string into an array |
| 5 | Join recipes with ~1.13M user reviews |
| 6 | Create time-based features |
| 7 | Save as Parquet |

## 1. Setup

In [1]:
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql import functions as F

# Import for typecasting columns
from pyspark.sql.types import IntegerType,BooleanType,DateType,FloatType,StringType
from pyspark.sql.types import ArrayType

In [3]:
spark = SparkSession.builder.appName("Recipe Recommender").getOrCreate()

In [4]:
spark

## 2. Load the Recipe Data
The raw recipe file is read from S3. Validation checks confirm the expected row count (231,637), column count and data types.

In [5]:
recipe_df = spark.read.options(header=True, inferSchema=True).csv('s3://foodproj/e-2RYFIQJPQ6ML5ST9TYHE146YH/RAW_recipes_cleaned.csv')

In [6]:
recipe_df.show(5)

+--------------------+------+-------+--------------+-------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-------------+
|                name|    id|minutes|contributor_id|          submitted|                tags|           nutrition|n_steps|               steps|         description|         ingredients|n_ingredients|
+--------------------+------+-------+--------------+-------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-------------+
|arriba   baked wi...|137739|     55|         47892|2005-09-16 00:00:00|['60-minutes-or-l...|[51.5, 0.0, 13.0,...|     11|['make a choice a...|autumn is my favo...|['winter squash',...|            7|
|a bit different  ...| 31490|     30|         26278|2002-06-17 00:00:00|['30-minutes-or-l...|[173.4, 18.0, 0.0...|      9|['preheat oven to...|this recipe calls...|['prepared pizza ...|            6|


In [7]:
assert recipe_df.count() == 231637, "There is a mistake in reading the data."
assert len(recipe_df.columns) == 12, "There is a mistake in reading the data."
assert recipe_df.schema["minutes"].dataType == IntegerType(), "The data types have not been read correctly."
assert recipe_df.schema["tags"].dataType == StringType(), "The data types have not been read correctly."
assert recipe_df.schema["n_ingredients"].dataType == IntegerType(), "The data types have not been read correctly."

## 3. Split the Nutrition Column
The `nutrition` column stores seven values as one string (e.g. `[51.5, 0.0, 13.0, ...]`). The brackets are removed and the values are split into individual numeric columns: calories plus six nutrients as % daily value (PDV).

In [8]:
# We can remove the Square brackets by doing the following code

recipe_df = (recipe_df.withColumn('nutrition',F.regexp_replace('nutrition', '\\[|\\]', '')))

In [9]:
# Now we can Split the nutrition columns, lets name it as nc_split
nc_split= F.split("nutrition",",")

# Extraction of individual names from the column, lets go with nc_name
nc_name= ["calories", "total_fat_PDV", "sugar_PDV", "sodium_PDV", "protein_PDV", "saturated_fat_PDV", "carbohydrates_PDV"]

for col_index, col_name in enumerate (nc_name):
    recipe_df= (recipe_df.withColumn(col_name,nc_split.getItem(col_index).cast(FloatType())))

In [10]:
recipe_df.show(1)

+--------------------+------+-------+--------------+-------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-------------+--------+-------------+---------+----------+-----------+-----------------+-----------------+
|                name|    id|minutes|contributor_id|          submitted|                tags|           nutrition|n_steps|               steps|         description|         ingredients|n_ingredients|calories|total_fat_PDV|sugar_PDV|sodium_PDV|protein_PDV|saturated_fat_PDV|carbohydrates_PDV|
+--------------------+------+-------+--------------+-------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-------------+--------+-------------+---------+----------+-----------+-----------------+-----------------+
|arriba   baked wi...|137739|     55|         47892|2005-09-16 00:00:00|['60-minutes-or-l...|51.5, 0.0, 13.0, ...|     11|['

In [11]:
# Validation check

assert recipe_df.schema["carbohydrates_PDV"].dataType == FloatType(), "Recheck your typecasting"
assert recipe_df.collect()[123432][14] == 62.0, "The columns have not been split correctly."
assert recipe_df.collect()[10000][12] == 60.400001525878906, "The columns have not been split correctly."

## 4. Standardise Nutrition per 100 Calories
PDV values depend on serving size, so each nutrient is also expressed **per 100 calories**. This makes recipes with very different portion sizes comparable. Recipes with zero calories would produce nulls, so those are filled with 0.

In [12]:
for nutrition_col in nc_name:
    if nutrition_col != 'calories': 
        nutrition_per_100_cal_col = nutrition_col.replace('_PDV', '') + '_per_100_cal'
        
        # Performing the transformation to calculate values per 100 calories
        recipe_df = recipe_df.withColumn(
            nutrition_per_100_cal_col,
            (F.col(nutrition_col) * 100 / F.col('calories'))
        )
        
        # Filling nulls with 0 in only the current nutrition_per_100_cal_col
        recipe_df = recipe_df.fillna({nutrition_per_100_cal_col: 0})

In [13]:
len(recipe_df.columns)

25

In [14]:
recipe_df.show(5)

+--------------------+------+-------+--------------+-------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-------------+--------+-------------+---------+----------+-----------+-----------------+-----------------+---------------------+------------------+------------------+-------------------+-------------------------+-------------------------+
|                name|    id|minutes|contributor_id|          submitted|                tags|           nutrition|n_steps|               steps|         description|         ingredients|n_ingredients|calories|total_fat_PDV|sugar_PDV|sodium_PDV|protein_PDV|saturated_fat_PDV|carbohydrates_PDV|total_fat_per_100_cal| sugar_per_100_cal|sodium_per_100_cal|protein_per_100_cal|saturated_fat_per_100_cal|carbohydrates_per_100_cal|
+--------------------+------+-------+--------------+-------------------+--------------------+--------------------+-------+--------------------+---------

In [15]:
# Total fat check for id 28881
assert recipe_df.filter("id == 28881").select('total_fat_per_100_cal').first()[0] == 0, "total_fat_per_100_cal for recipe 28881 should be 0"

# Total fat check for id 112140
assert round(recipe_df.filter("id == 112140").select('total_fat_per_100_cal').first()[0]) == 8, "total_fat_per_100_cal for recipe 112140 should be 8"

# Checking for nulls
for c in ['total_fat_per_100_cal', 'sugar_per_100_cal', 'sodium_per_100_cal', 
          'protein_per_100_cal', 'saturated_fat_per_100_cal', 'carbohydrates_per_100_cal']:
    assert recipe_df.select(F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c)).collect()[0][0] == 0, "There are Nulls in the data"


## 5. Convert Tags to an Array
Tags arrive as a single string like `"['60-minutes-or-less', 'easy']"`. Brackets and quotes are stripped and the string is split into a proper array so individual tags can be analysed later.

In [16]:
print(recipe_df)

DataFrame[name: string, id: int, minutes: int, contributor_id: int, submitted: timestamp, tags: string, nutrition: string, n_steps: int, steps: string, description: string, ingredients: string, n_ingredients: int, calories: float, total_fat_PDV: float, sugar_PDV: float, sodium_PDV: float, protein_PDV: float, saturated_fat_PDV: float, carbohydrates_PDV: float, total_fat_per_100_cal: double, sugar_per_100_cal: double, sodium_per_100_cal: double, protein_per_100_cal: double, saturated_fat_per_100_cal: double, carbohydrates_per_100_cal: double]

In [17]:
from pyspark.sql.functions import regexp_replace, split
     # pyspark function to remove symbols like '[' ']' "'" from the strings in the tags column.
     # pyspark function to split the column using the comma delimiter.
        
recipe_df = (recipe_df
                  .withColumn('tags', (F.regexp_replace("tags","[\[\]'']", "")))
                  .withColumn('tags', split(F.col("tags"), ", ").cast(ArrayType(StringType())).alias("tags")))

In [18]:
recipe_df.show()

+--------------------+------+-------+--------------+-------------------+--------------------+--------------------+-------+--------------------+--------------------+--------------------+-------------+--------+-------------+---------+----------+-----------+-----------------+-----------------+---------------------+------------------+------------------+-------------------+-------------------------+-------------------------+
|                name|    id|minutes|contributor_id|          submitted|                tags|           nutrition|n_steps|               steps|         description|         ingredients|n_ingredients|calories|total_fat_PDV|sugar_PDV|sodium_PDV|protein_PDV|saturated_fat_PDV|carbohydrates_PDV|total_fat_per_100_cal| sugar_per_100_cal|sodium_per_100_cal|protein_per_100_cal|saturated_fat_per_100_cal|carbohydrates_per_100_cal|
+--------------------+------+-------+--------------+-------------------+--------------------+--------------------+-------+--------------------+---------

In [19]:
recipe_df.printSchema()

root
 |-- name: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- minutes: integer (nullable = true)
 |-- contributor_id: integer (nullable = true)
 |-- submitted: timestamp (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- nutrition: string (nullable = true)
 |-- n_steps: integer (nullable = true)
 |-- steps: string (nullable = true)
 |-- description: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- n_ingredients: integer (nullable = true)
 |-- calories: float (nullable = true)
 |-- total_fat_PDV: float (nullable = true)
 |-- sugar_PDV: float (nullable = true)
 |-- sodium_PDV: float (nullable = true)
 |-- protein_PDV: float (nullable = true)
 |-- saturated_fat_PDV: float (nullable = true)
 |-- carbohydrates_PDV: float (nullable = true)
 |-- total_fat_per_100_cal: double (nullable = false)
 |-- sugar_per_100_cal: double (nullable = false)
 |-- sodium_per_100_cal: double (nullable = false)
 |-

In [20]:
# Validation check

assert recipe_df.schema["tags"].dataType == ArrayType(StringType(), True), "You have not split the string into an array."
assert recipe_df.collect()[2][5] == ['time-to-make','course', 'preparation', 'main-dish', 'chili', 'crock-pot-slow-cooker', 'dietary', 'equipment', '4-hours-or-less'], "Recheck your string cleaning and splitting operations."


## 6. Join Recipes with User Reviews
The interactions file (one row per user review) is loaded and left-joined to the recipe data, so each review carries the full recipe details.

In [21]:
raw_ratings_df = (spark.read.csv("s3a://foodproj/e-2RYFIQJPQ6ML5ST9TYHE146YH/RAW_interactions_cleaned.csv", 
                                 header=True, 
                                 inferSchema=True)
                  .withColumn("review_date", F.col("date"))
                  .drop("date"))

In [22]:
# lets ensure it has joined correctly 
raw_ratings_df.printSchema()
raw_ratings_df.show(5)

root
 |-- user_id: integer (nullable = true)
 |-- recipe_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- review: string (nullable = true)
 |-- review_date: timestamp (nullable = true)

+-------+---------+------+--------------------+-------------------+
|user_id|recipe_id|rating|              review|        review_date|
+-------+---------+------+--------------------+-------------------+
|  38094|    40893|     4|Great with a sala...|2003-02-17 00:00:00|
|1293707|    40893|     5|So simple  so del...|2011-12-21 00:00:00|
|   8937|    44394|     4|This worked very ...|2002-12-01 00:00:00|
| 126440|    85009|     5|I made the Mexica...|2010-02-27 00:00:00|
|  57222|    85009|     5|Made the cheddar ...|2011-10-01 00:00:00|
+-------+---------+------+--------------------+-------------------+
only showing top 5 rows

In [23]:
# Validation check

assert raw_ratings_df.count() == 1132367, "There is a mistake in reading the data."
assert len(raw_ratings_df.columns) == 5, "There is a mistake in reading the data."

In [32]:
interaction_level_df = raw_ratings_df.join(
    recipe_df,  # Joining with the recipes DataFrame
    raw_ratings_df.recipe_id == recipe_df.id,
    "left") 

In [33]:
# Validation check

assert (interaction_level_df.count() ,len(interaction_level_df.columns)) == (1132367, 30), "The type of join is incorrect"

list1 = raw_ratings_df.select('recipe_id').collect()
list2 = recipe_df.select('id').collect()
exclusive_set = set(list1)-set(list2)

assert len(exclusive_set) == 0, "There is a mistake in reading one of the two data files."

In [35]:
interaction_level_df.count(), len(interaction_level_df.columns)

(1132367, 30)

**Result:** 1,132,367 reviews × 30 columns. Every review matched a recipe.

## 7. Time-Based Features
For each review, the time between when the recipe was **submitted** and when it was **reviewed** is calculated in days, months and years. This lets us test whether older recipes are rated differently.

In [36]:
# for the following task, we can convert the 'submitted' and 'review_date' columns to DateType, 
# i would also rename "interaction_level_df" as idf for now

idf= interaction_level_df

In [39]:
# just a quick check

idf.count(), len(idf.columns)

(1132367, 30)

In [44]:
# Convert the 'submitted' and 'review_date' columns to DateType
idf = (idf
       .withColumn('submitted', F.col('submitted').cast(DateType()))
       .withColumn('review_date', F.col('review_date').cast(DateType())))

# Calculate the number of days between the submission date and the review date
idf = idf.withColumn('days_since_submission_on_review_date',
                     F.datediff(F.col('review_date'), F.col('submitted')))

# Calculate the number of months between the submission date and the review date
idf = idf.withColumn('months_since_submission_on_review_date',
                     F.months_between(F.col('review_date'), F.col('submitted')))

# Calculate the number of years between the submission date and the review date
idf = idf.withColumn('years_since_submission_on_review_date',
                     F.col('months_since_submission_on_review_date') / 12)

In [47]:
# we can change idf back to its full name for the code check
interaction_level_df = idf

In [48]:
# Validation check

assert interaction_level_df.schema["days_since_submission_on_review_date"].dataType == IntegerType()

assert (interaction_level_df.filter((interaction_level_df.user_id == 428885) & (interaction_level_df.recipe_id == 335241))
                            .select('days_since_submission_on_review_date').collect()[0][0]) == 77
assert (interaction_level_df.filter((interaction_level_df.user_id == 2025676) & (interaction_level_df.recipe_id == 94265))
                            .select('months_since_submission_on_review_date').collect()[0][0]) == 153.22580645
assert (interaction_level_df.filter((interaction_level_df.user_id == 338588) & (interaction_level_df.recipe_id == 21859))
                            .select('years_since_submission_on_review_date').collect()[0][0]) == 4.564516129166667

## 8. Save the Processed Data
The interaction-level dataset (1,132,367 rows × 33 columns) is written to S3 as Parquet for the EDA notebook.

In [49]:
interaction_level_df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- recipe_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- review: string (nullable = true)
 |-- review_date: date (nullable = true)
 |-- name: string (nullable = true)
 |-- id: integer (nullable = true)
 |-- minutes: integer (nullable = true)
 |-- contributor_id: integer (nullable = true)
 |-- submitted: date (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- nutrition: string (nullable = true)
 |-- n_steps: integer (nullable = true)
 |-- steps: string (nullable = true)
 |-- description: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- n_ingredients: integer (nullable = true)
 |-- calories: float (nullable = true)
 |-- total_fat_PDV: float (nullable = true)
 |-- sugar_PDV: float (nullable = true)
 |-- sodium_PDV: float (nullable = true)
 |-- protein_PDV: float (nullable = true)
 |-- saturated_fat_PDV: float (nullable = true)
 |-- carboh

In [50]:
assert (interaction_level_df.count() ,len(interaction_level_df.columns) ) == (1132367, 33)

In [51]:
interaction_level_df.write \
    .mode('overwrite') \
    .parquet('s3a://foodproj/e-2RYFIQJPQ6ML5ST9TYHE146YH/Processed Files/interaction_level_df_processed.parquet')